In [0]:
%run ../config/config

# dim_source and dim_asset

Both tables are filled with insert-only `MERGE`: new rows are added, existing rows are not changed.
Surrogate keys stay the same between runs, and a rerun without new data changes nothing.

- `dim_source`: two sources - `kraken_api` (batch) and `streaming` (CoinGecko via Event Hub).
- `dim_asset`: one row per symbol found in silver (current OHLC versions and ticks), with a readable name.
  A symbol that appears in the data later is added automatically; without a known name it gets `unknown`.

In [0]:
from utilities.gold_transformations import (
    build_dim_asset,
    current_ohlc,
    dim_source_rows,
    distinct_symbols,
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

ohlc = current_ohlc(spark.table(SILVER_TABLE))
ticks = spark.table(SILVER_TABLE)

### dim_source

In [0]:
(spark.createDataFrame(dim_source_rows(), "source_name string, source_type string, provider string")
    .createOrReplaceTempView("dim_source_src"))

spark.sql(f"""
MERGE INTO {GOLD}.dim_source t
USING dim_source_src s
ON t.source_name = s.source_name
WHEN NOT MATCHED THEN INSERT (source_name, source_type, provider)
VALUES (s.source_name, s.source_type, s.provider)
""")

### dim_asset

In [0]:
build_dim_asset(distinct_symbols(ohlc, ticks)).createOrReplaceTempView("dim_asset_src")

spark.sql(f"""
MERGE INTO {GOLD}.dim_asset t
USING dim_asset_src s
ON t.symbol = s.symbol
WHEN NOT MATCHED THEN INSERT (symbol, asset_name)
VALUES (s.symbol, s.asset_name)
""")